In [1]:
!pip -q install --upgrade huggingface_hub 
!apt -q install git -y

Reading package lists...
Building dependency tree...
Reading state information...
git is already the newest version (1:2.17.1-1ubuntu0.18).
0 upgraded, 0 newly installed, 0 to remove and 58 not upgraded.


In [1]:
from huggingface_hub import login

login(token="")

In [10]:
with open("/mario-gpt/notebooks/test_level.txt", "r") as file:
    file_contents = file.read()

print(file_contents)

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
-----##--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
SSSSS##SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS------------SSSSSSSSSSSSSSSSSSSSSS------------SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS-------------SSSSSSSSSSSSSSSSSS
------------------------SSSSSSSSSSSSSS------------------------------------SS---SS----------------------------------------------------------------------------------------SSSS----------------------------------------------------------------------------


In [11]:
def transform_to_list(input_string):
    # Split the input string by newline characters
    lines = input_string.strip().split('\n')
    return lines

file_contents = transform_to_list(file_contents)

In [12]:
for line in file_contents:
    print(line)

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
-----##--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
SSSSS##SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS------------SSSSSSSSSSSSSSSSSSSSSS------------SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS-------------SSSSSSSSSSSSSSSSSS
------------------------SSSSSSSSSSSSSS------------------------------------SS---SS----------------------------------------------------------------------------------------SSSS----------------------------------------------------------------------------


In [13]:
type(file_contents)

list

In [14]:
from mario_gpt.prompter import Prompter
from mario_gpt.prompt_adapter import PromptAdapter
from transformers import pipeline

# Pre-initialize the LLM model
llm_model = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct", device=0)


In [15]:
from transformers import AutoTokenizer

# Create a Prompter instance
tokenizer_path = "shyamsn97/Mario-GPT2-700-context-length"
level_tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
prompter = Prompter(level_tokenizer=level_tokenizer) 

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [16]:
tokenized_level = prompter.level_tokenizer(file_contents, return_tensors="pt")

level_tensor = tokenized_level['input_ids']
flattened_tensor = level_tensor.view(-1)

prompt_base, _, _, _ = prompter(level=flattened_tensor)  # Generate the structured prompt
print(prompt_base)

Enemy Count: 19, Power-Up Count: 6
0 pipes, 0 enemys, 832 blocks, 8 koopas, 11 goombas, 9 powerups, 39 coins, Hard, high elevation


In [9]:
# Create a PromptAdapter instance
adapter = PromptAdapter(llm=llm_model, prompter=prompter)

# Generate a new adapted prompt
level_data = file_contents
adapted_prompt = adapter.new_prompter(level_data)

print("Adapted Prompt:", adapted_prompt)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


tensor([13, 13, 13,  ..., 56, 56, 56])
no pipes, no enemies, many blocks, many goombas, little koopas, little coins, little powerups, high elevation
Prompt for adapter:  
                
    An Increased Reasoning Steps Prompt
    I want you to act as a Prompt Rewriter.
    Your objective is to rewrite a given prompt into a version that requires multiple-step reasoning.
    If the given prompt can be solved with just a few simple thinking processes, you can rewrite it to explicitly request multiple-step reasoning.
    You should try your best not to make the evolved prompt verbose, and the evolved prompt can only add 10 to 20 words.
    'Given Prompt', 'Evolved Prompt', 'given prompt' and 'evolved prompt' are not allowed to appear in the Evolved Prompt


                ### Given Prompt:
                Generate levels with no pipes, no enemies, many blocks, many goombas, little koopas, little coins, little powerups, high elevation

                ### Evolved Prompt:
                